# 01 — Nấc 1: Inference thông thường

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/vinumqa-numerical-reasoning/blob/main/notebooks/01_baseline_plain.ipynb)

**Cần GPU** (A100 khuyến nghị; L4 cũng được). Khoảng 20–30 phút.

## Mục đích

Đây là **mốc dưới** của toàn bộ lộ trình: Qwen3-8B, không prompt engineering, không
fine-tune, không self-eval, không ACE. Chỉ đưa ngữ cảnh + câu hỏi và yêu cầu trả lời
bằng một chương trình tính toán.

Mọi con số ở bốn nấc sau đều được đo **so với nấc này**, nên nó phải được chạy đúng và
lưu lại cẩn thận.

## Prompt ở nấc này

Tối giản có chủ đích — chỉ nêu tác vụ, liệt kê tên các phép toán, và định dạng đầu ra.
**Không có**: giải thích từng phép toán, ánh xạ từ khoá tiếng Việt → phép toán, quy tắc
xử lý đơn vị, ví dụ mẫu. Toàn bộ những thứ đó là nội dung của nấc 2, và hiệu số giữa hai
nấc chính là giá trị của prompt engineering.

> Vẫn phải yêu cầu đúng định dạng `program:` / `answer:`, nếu không thì không chấm được
> gì cả. Đây là mức tối giản *đo được*, không phải prompt trần trụi.

## Tham số

Giữ nguyên của `reference/original_notebooks/inference_with_difference_models.ipynb`:
`unsloth/Qwen3-8B`, `load_in_4bit=True`, `fast_inference=True` (vLLM),
`temperature=0.1`, `max_tokens=3000`.

## §1. Môi trường

In [ ]:
# Cài đặt — ghim theo bộ ĐÃ XÁC MINH cài xong sạch trên image Colab hiện tại
# (Python 3.13, torch 2.11.0+cu128, A100).
#
# ⚠ KHÁC bản tham chiếu, và đây là chủ ý:
#   Khối cài đặt gốc ghim transformers==4.56.2 / trl==0.22.2 / xformers==0.0.29.post3.
#   Trên image Colab hiện tại nó THẤT BẠI — nhánh chọn xformers chỉ biết torch 2.8/2.9,
#   gặp torch 2.11 thì rơi vào bản 0.0.29.post3 (dành cho torch 2.5) nên đổ cả khối,
#   mà `%%capture` lại nuốt mất báo lỗi.
#   Bộ dưới đây là bộ pip tự giải ra khi để `unsloth` và `vllm` thoả thuận với nhau.
#   Chênh lệch phiên bản được ghi vào `env` của meta mỗi nấc, nên báo cáo vẫn truy được.
#
# ⏱ 6–12 phút (đã ghim nên pip khỏi dò tìm). Cố ý KHÔNG giấu output để thấy nó còn sống.
import os, time
_t_cai = time.time()
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install unsloth==2026.9.4 transformers==4.57.6 trl==0.24.0 peft==0.20.0 bitsandbytes==0.50.2 xformers==0.0.35
    # vLLM phải khớp CUDA của torch. Bản trên PyPI dựng cho CUDA 13, còn Colab đang
    # CUDA 12.8 → unsloth CHẶN import và báo "No module named 'vllm'" dù gói vẫn có.
    # Wheel dưới đây là bản cu129, đúng cái unsloth khuyến nghị cho hệ CUDA 12.x.
    # Nếu image Colab đổi CUDA: chạy ô này, đọc dòng WARNING của unsloth ở cell sau —
    # nó in ra đúng URL wheel cần dùng, thay vào đây là xong.
    !pip install https://github.com/vllm-project/vllm/releases/download/v0.23.0/vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl
print(f"\n[CÀI ĐẶT] xong sau {(time.time() - _t_cai) / 60:.1f} phút")
print("[CÀI ĐẶT] Colab hiện nút RESTART SESSION thì bấm, rồi chạy lại TỪ CELL #2 "
      "(bỏ qua ô này — cài lại là thừa).")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
ON_KAGGLE = os.path.isdir("/kaggle/input")
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
elif ON_KAGGLE:
    for _r, _d, _f in os.walk("/kaggle/input"):
        if "vinumqa" in _d and "data" in _d:
            REPO_DIR = _r; _pinned = True; break
    OUTPUT_DIR = "/kaggle/working/vinumqa_runs"
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, sft, stats
from vinumqa.prompts import PromptKit

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

LADDER = [
    ("01_plain",           "Nấc 1 — inference thông thường"),
    ("02_prompt_eng",      "Nấc 2 — + prompt engineering"),
    ("02b_no_fewshot",     "Nấc 2b — prompt bỏ 2 ví dụ mẫu"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("04_selfeval_sft",    "Nấc 4b — + self-eval (model SFT)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05_ace_sft",         "Nấc 5b — + ACE (model SFT)"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("06_comb_F_A",        "Tổ hợp — SFT + ACE (không self-eval)"),
]
LADDER_LABEL = dict(LADDER)

# Những biến phải đi kèm kết quả thì mới truy lại được về sau.
_CFG_KEYS = ("MODEL_NAME", "MODEL_TAG", "TEMPERATURE", "MAX_TOKENS", "REPETITION_PENALTY",
             "MAX_SEQ_LENGTH", "BATCH_SIZE", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "RANDOM_SEED")


def run_env():
    """Môi trường THẬT lúc chạy: commit, GPU, phiên bản thư viện.

    Chỉ đọc thư viện đã nạp (``sys.modules``) chứ không import thêm — vừa nhanh,
    vừa báo đúng những gì thật sự được dùng.
    """
    env = {"python": sys.version.split()[0]}
    try:                                   # bản code nào sinh ra kết quả này
        def _g(*a):
            return subprocess.run(["git", "-C", REPO_DIR, *a],
                                  capture_output=True, text=True).stdout.strip()
        env["commit"] = _g("rev-parse", "--short", "HEAD")
        env["branch"] = _g("rev-parse", "--abbrev-ref", "HEAD")
        env["dirty"] = bool(_g("status", "--porcelain"))
    except Exception:
        pass
    _torch = sys.modules.get("torch")
    if _torch is not None:
        env["torch"] = getattr(_torch, "__version__", "?")
        try:
            if _torch.cuda.is_available():
                _p = _torch.cuda.get_device_properties(0)
                env["gpu"] = _p.name
                env["vram_gb"] = round(_p.total_memory / 1024**3, 1)
                env["cc"] = f"{_p.major}.{_p.minor}"
            else:
                env["gpu"] = "CPU"
        except Exception:
            pass
    else:
        env["gpu"] = "CPU (không nạp torch)"
    for _lib in ("transformers", "trl", "peft", "vllm", "unsloth",
                 "sentence_transformers", "numpy"):
        _m = sys.modules.get(_lib)
        if _m is not None and hasattr(_m, "__version__"):
            env[_lib] = _m.__version__
    return env


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, csv, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "csv":   f"{stage}_program.csv",
        "meta":  f"{stage}_meta.json"}[kind])


MAX_TOKENS_CHUAN = 3000          # trần sinh của cả thang bậc


def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: 3 file chuẩn + 1 file output thô.

    Chạy với ``MAX_TOKENS`` khác mức chuẩn thì tự ghi sang tên nấc khác. Đổi trần sinh
    là đổi cấu hình, kết quả không so thẳng với thang bậc được — mà nếu cứ ghi đè lên
    tên cũ thì mất luôn bản chuẩn, không lấy lại được nếu không chạy lại GPU.
    """
    _mt = globals().get("MAX_TOKENS")
    if _mt and _mt != MAX_TOKENS_CHUAN and not stage.endswith(f"_tok{_mt}"):
        stage = f"{stage}_tok{_mt}"
        if not quiet:
            print(f"[GHI] ⚠ MAX_TOKENS={_mt} ≠ {MAX_TOKENS_CHUAN} → ghi sang nấc "
                  f"'{stage}', không đè lên kết quả chuẩn.")
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_details_csv(rows, stage_path(stage, "csv"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            "enable_thinking": getattr(globals().get("prompt_kit", None),
                                       "enable_thinking", "?"),
            "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
            "env": run_env(),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'csv')}   ← 6 cột, mở bằng Excel được")
        print(f"      {stage_path(stage, 'meta')}")


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else ("Kaggle" if ON_KAGGLE else "local")
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + csv + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

In [ ]:
# ═══ Self-test: chạy TRƯỚC khi tốn GPU ═══
# Cell này đỏ thì dừng lại — mọi con số PA/EA sau đó sẽ vô nghĩa.

# (1) Ô cài đặt có thật sự cài được không. Đọc metadata nên nhanh, không phải import.
#     Kiểm ở đây để lỗi pip lộ ra trong 1 giây, thay vì 20 phút nữa lúc nạp model.
from importlib.metadata import version as _ver, PackageNotFoundError as _NoPkg
_goi = {}
for _p in ("vllm", "unsloth", "transformers", "trl", "peft", "torch"):
    try:
        _goi[_p] = _ver(_p)
    except _NoPkg:
        _goi[_p] = None
print("[GÓI] " + " | ".join(f"{k}={v}" for k, v in _goi.items() if v))
# vLLM phải khớp CUDA của torch, nếu không unsloth CHẶN import dù gói vẫn có mặt —
# lúc đó cell nạp model báo "No module named 'vllm'" một cách khó hiểu.
# Wheel khớp CUDA có đuôi "+cuXXX" trong số phiên bản; bản PyPI thì không.
if _goi.get("vllm") and "+cu" not in _goi["vllm"]:
    print("[GÓI] ⚠ vllm=" + _goi["vllm"] + " là bản PyPI (dựng cho CUDA 13). Nếu cell nạp "
          "model báo \"No module named 'vllm'\" thì cài lại bằng wheel khớp CUDA — "
          "dòng WARNING của unsloth in sẵn URL đúng.")

_thieu = [k for k, v in _goi.items() if v is None]
if _thieu:
    raise RuntimeError(
        "Thiếu gói: " + ", ".join(_thieu) + " — ô cài đặt (cell #1) đã thất bại.\n\n"
        "Cách chữa: mở Cửa sổ dòng lệnh (góc dưới trái), chạy\n"
        "    pip install -U unsloth vllm\n"
        "xem lỗi thật, xong Restart session rồi chạy lại TỪ CELL #2 (bỏ qua cell #1).")

# vLLM 0.23 cấm toàn bộ transformers 5.x. Gói nào đó nâng lên 5 thì chặn ngay tại đây,
# đừng để phát hiện sau 4 phút nạp model. (sentence-transformers ≥ 6 là thủ phạm hay gặp.)
if str(_goi["transformers"]).split(".")[0] != "4":
    raise RuntimeError(
        "transformers=" + str(_goi["transformers"]) + " — vLLM 0.23 chỉ chạy với "
        "transformers 4.x, gói nào đó đã nâng nó lên.\n"
        "Chữa: pip install \"transformers==4.57.6\" rồi Restart session.")

# (2) Executor có tái tạo đúng nhãn vàng không.
_ok = sum(dsl.check_ea(dsl.execute_program(s["qa"]["program"], s.get("table") or []),
                       s["qa"].get("exe_ans")) for s in test_all)
print(f"[SELF-TEST] executor tái tạo exe_ans trên test: {_ok}/{len(test_all)}")
assert _ok / len(test_all) > 0.99, "Executor không tái tạo được nhãn vàng — DỪNG."
assert dsl.execute_program("divide(5310, add(1, 0.15))", []) is None   # lồng nhau
assert dsl.check_pa("add(1, 2)", "add(2, 1)")[0]                       # giao hoán
assert dsl.check_ea(0.6066481994, "0.60665")                           # làm tròn 5 chữ số
print("[SELF-TEST] ✅ executor / PA / EA đạt")

## §2. Model

In [ ]:
# ═══════════════ MODEL — Qwen3-8B 4-bit ═══════════════
# Tham số lấy từ reference/original_notebooks/inference_with_difference_models.ipynb:
#   load_in_4bit=True, fast_inference=True, temperature=0.1, max_tokens=3000
MODEL_NAME = "unsloth/Qwen3-8B"
MODEL_TAG  = "Qwen3-8B"

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Không thấy GPU. Runtime → Change runtime type → L4 GPU.")
_GPU, _VRAM = torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1024**3
_CC = torch.cuda.get_device_capability(0)
if _CC[0] < 7:
    raise RuntimeError(f"{_GPU} (CC {_CC[0]}.{_CC[1]}) không chạy được vLLM. "
                       f"Trên Kaggle chọn 'GPU T4 x2', đừng chọn P100.")

TEMPERATURE, MAX_TOKENS = 0.1, 3000        # giữ đúng mốc tham chiếu
REPETITION_PENALTY = 1.0
BATCH_SIZE = 500                           # như notebook cũ; giảm nếu OOM

if _VRAM < 18:                             # T4 15GB
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 8192, 0.90, 16
    MAX_TOKENS, BATCH_SIZE = 1536, 32
elif _VRAM < 30:                           # L4 24GB
    # 13500 chứ không phải 12000: để ngân sách prompt (13500-3000=10500) vượt xa
    # prompt bước 2 dài nhất (~9k token) → L4 KHÔNG phải cắt ngữ cảnh, nhờ vậy
    # kết quả trên L4 và A100 so sánh trực tiếp được với nhau.
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 13500, 0.88, 24
    BATCH_SIZE = 128
elif _VRAM < 60:                           # A100 40GB — như notebook gốc
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 15000, 0.85, 64
else:                                      # A100 80GB — còn dư, tăng song song
    MAX_SEQ_LENGTH, GPU_MEM_UTIL, MAX_NUM_SEQS = 15000, 0.85, 128
DTYPE = torch.float16 if _CC[0] < 8 else None

print(f"[GPU] {_GPU} | {_VRAM:.1f} GB | CC {_CC[0]}.{_CC[1]}")
print(f"[CFG] max_seq={MAX_SEQ_LENGTH} max_tokens={MAX_TOKENS} temp={TEMPERATURE} "
      f"batch={BATCH_SIZE}")
if _VRAM < 18:
    print("[CFG] ⚠ T4: đã hạ max_tokens xuống 1536 → kết quả KHÔNG so trực tiếp "
          "được với máy dùng 3000. Nên chạy trên L4.")

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import unsloth
from unsloth import FastLanguageModel
from vllm import SamplingParams

torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)

import shutil as _sh

# Đặt True nếu model tải về bị thiếu trọng số: tắt hf_transfer thì tải chậm hơn vài phút
# nhưng có kiểm tra và tải tiếp được. Lưu ý: `export` trong Cửa sổ dòng lệnh KHÔNG tới
# được kernel notebook — phải đặt ở đây.
TAI_CHAM_CHO_CHAC = False
if TAI_CHAM_CHO_CHAC:
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    print("[MODEL] đã tắt hf_transfer — tải chậm hơn nhưng chắc hơn")

_free = _sh.disk_usage("/").free / 1024**3
_t_nap = time.time()
print(f"[MODEL] Đang tải {MODEL_NAME} ... (đĩa trống {_free:.0f} GB)")
if _free < 15:
    print("[MODEL] ⚠ dưới 15 GB trống — model ~15 GB, tải dễ đứt giữa chừng.")
print("[MODEL] ⏳ Mất 4–7 PHÚT. Tải xong rồi vLLM còn dựng CUDA graph — đoạn đó")
print("[MODEL]    KHÔNG có thanh tiến trình, nhìn như treo nhưng không phải.")
print("[MODEL]    Muốn biết còn sống: xem MỐC GIỜ ở các dòng INFO bên dưới. Nó nhích")
print("[MODEL]    lên là đang chạy. Đứng im quá 10 phút mới đáng nghi.")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name             = MODEL_NAME,
        dtype                  = DTYPE,
        max_seq_length         = MAX_SEQ_LENGTH,
        load_in_4bit           = True,
        fast_inference         = True,
        gpu_memory_utilization = GPU_MEM_UTIL,
        max_num_seqs           = MAX_NUM_SEQS,
    )
except (RuntimeError, ValueError) as _e:
    # Thiếu trọng số một vài lớp = shard safetensors tải dở còn nằm trong cache.
    # vLLM không tự sửa được, phải xoá cache tải lại.
    if "not initialized from checkpoint" not in str(_e):
        raise
    raise RuntimeError(
        "Model thiếu trọng số — bản tải dở trong cache HuggingFace.\n\n"
        "Bước 1 — xoá cache. Mở Cửa sổ dòng lệnh (góc dưới trái):\n"
        "    rm -rf ~/.cache/huggingface/hub/models--unsloth--Qwen3-8B*\n"
        "    df -h / | tail -1          # kiểm luôn, cần ≥ 20 GB trống\n\n"
        "Bước 2 — đặt TAI_CHAM_CHO_CHAC = True ở ĐẦU CHÍNH Ô NÀY.\n"
        "    (`export` trong terminal không tới được kernel notebook.)\n\n"
        "Bước 3 — Restart session, chạy lại TỪ CELL #2 (bỏ qua ô cài đặt).\n\n"
        "Hỏng y hệt lần nữa thì không phải do tải: khi đó là bản 4-bit của unsloth "
        "không khớp bộ nạp của vLLM, phải đổi phiên bản chứ không phải tải lại.") from _e

SAMPLING = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                          repetition_penalty=REPETITION_PENALTY, seed=RANDOM_SEED)
LORA_REQUEST = None          # nấc 3 trở đi có thể gán adapter đã SFT vào đây

def generate(prompts, sampling_params=None, desc=None, batch_size=None):
    """Sinh theo lô qua vLLM — như vòng lặp trong notebook cũ."""
    if not prompts:
        return []
    sp = sampling_params or SAMPLING
    bs = batch_size or BATCH_SIZE
    outs, t0 = [], time.time()
    nb = (len(prompts) + bs - 1) // bs
    for i in range(nb):
        chunk = prompts[i*bs:(i+1)*bs]
        kw = {"sampling_params": sp}
        if LORA_REQUEST is not None:
            kw["lora_request"] = LORA_REQUEST
        outs.extend(o.outputs[0].text for o in model.fast_generate(chunk, **kw))
        if desc:
            el = time.time() - t0
            print(f"    {desc}: lô {i+1}/{nb} | {el:.0f}s | "
                  f"ETA {el/(i+1)*(nb-i-1):.0f}s", end="\r")
    gc.collect(); torch.cuda.empty_cache()
    if desc:
        print(f"    {desc}: xong {len(prompts)} prompt trong {time.time()-t0:.0f}s" + " "*16)
    return outs

_t = torch.cuda.get_device_properties(0).total_memory/1024**3
print(f"[MODEL] ✅ sẵn sàng sau {(time.time()-_t_nap)/60:.1f} phút | "
      f"VRAM {_t - torch.cuda.mem_get_info()[0]/1024**3:.1f}/{_t:.1f} GB")
_ = generate(["xin chào"], SamplingParams(temperature=0, max_tokens=4))
print("[WARMUP] ✅")

## §3. Prompt tối giản

In [ ]:
prompt_kit = PromptKit(REPO_DIR, tokenizer=tokenizer, model_name=MODEL_NAME)
PROMPT_LEVEL = "plain"
USE_SELFEVAL = False
if PROMPT_LEVEL == "selfeval":
    PROMPT_LEVEL = "engineered"

_think = getattr(prompt_kit, "enable_thinking", None)
print(f"[PROMPT] mức = {PROMPT_LEVEL} | self-eval = {USE_SELFEVAL} | "
      f"suy nghĩ = {'template tự quyết (Qwen3: BẬT)' if _think is None else _think}")
if _think is False:
    print("[PROMPT] ⚠ suy nghĩ đang TẮT — lệch bản tham chiếu, PA sẽ hụt "
          "~10 điểm. Dấu hiệu: 497 mẫu chạy xong trong ~1 phút.")
print(f"         plain={len(prompt_kit.PLAIN_SYSTEM_PROMPT)} ký tự | "
      f"engineered={len(prompt_kit.ENGINEERED_SYSTEM_PROMPT)} | "
      f"self-eval={len(prompt_kit.SELF_EVAL_SYSTEM_PROMPT)}")

# Đo bằng tokenizer THẬT trên 40 mẫu có ngữ cảnh DÀI NHẤT
BUDGET = MAX_SEQ_LENGTH - MAX_TOKENS
_clen = lambda s: (len(" ".join(s.get("pre_text") or [])) +
                   len(" ".join(s.get("post_text") or [])) + len(str(s.get("table") or "")))
_probe = sorted(test_all, key=_clen, reverse=True)[:40]
_bul = "\n".join(["- Khi hỏi tốc độ tăng trưởng, dùng subtract(gia_tri_moi, gia_tri_cu), "
                  "divide(#0, gia_tri_cu)."] * 7)
# Lời giải bước 1 dài nhất có thể là đúng MAX_TOKENS token (model sinh chạm trần).
# Phải đo ở mức đó, không thì bật suy nghĩ vào là prompt bước 2 tràn ngân sách.
_unit = "Phân tích chi tiết từng bước của bảng số liệu. "
_prev = _unit * max(1, MAX_TOKENS // max(1, len(tokenizer(_unit).input_ids)))
_prev += "\n```plaintext\nprogram: divide(1,2)\nanswer: 0.5\n```"

def _measure():
    a = [len(tokenizer(prompt_kit.step1(s, _bul, level=PROMPT_LEVEL)).input_ids)
         for s in _probe]
    b = ([len(tokenizer(prompt_kit.step2(s, _prev, _bul)).input_ids) for s in _probe]
         if USE_SELFEVAL else [0])
    return a, b

_a, _b = _measure()
print(f"[PROMPT] (40 mẫu dài nhất) step1 max={max(_a)} | step2 max={max(_b)} | "
      f"ngân sách={BUDGET}")

if max(max(_a), max(_b)) > BUDGET:
    prompt_kit.max_prev_chars = 3000
    _cap = _clen(_probe[0])
    for _ in range(6):
        _cap = int(_cap * 0.80)
        prompt_kit.max_ctx_chars = max(1200, _cap)
        _a, _b = _measure()
        if max(max(_a), max(_b)) <= BUDGET:
            break
    assert max(max(_a), max(_b)) <= BUDGET, "Không cắt đủ — giảm MAX_TOKENS hoặc dùng GPU lớn hơn."
    _hit = sum(1 for s in test_all if _clen(s) > prompt_kit.max_ctx_chars)
    print(f"[PROMPT] ⚠ đã bật cắt ngữ cảnh (max_ctx_chars={prompt_kit.max_ctx_chars}); "
          f"{_hit}/{len(test_all)} mẫu bị cắt ({_hit/len(test_all)*100:.1f}%)")
    print(f"[PROMPT]   GHI LẠI con số này khi báo cáo.")
else:
    print("[PROMPT] ✅ mọi prompt đều lọt ngân sách, không cần cắt")

In [ ]:
# Xem thử prompt thật sự gửi cho model
_demo = prompt_kit.step1(test_all[0], level="plain")
print(_demo[:1800])
print("\n... [cắt] ...\n")
print(f"[PROMPT] tổng {len(tokenizer(_demo).input_ids)} token")

## §4. Chạy trên tập test

In [ ]:
# (phần đọc/ghi đã nằm ở cell đầu)

In [ ]:
STAGE = "01_plain"
print(f"\n{'═'*74}\n  NẤC: {STAGE} | prompt={PROMPT_LEVEL} | self-eval={USE_SELFEVAL}"
      f" | {len(test_all)} mẫu\n{'═'*74}")

_t0 = time.time()
rows = pipeline.run_pipeline(
    test_all, prompt_kit, generate,
    prompt_level=PROMPT_LEVEL, use_selfeval=USE_SELFEVAL,
    sp_step1=SAMPLING, sp_step2=SAMPLING, desc=STAGE)
metrics = pipeline.summarize(rows, STAGE)
metrics["minutes"] = round((time.time() - _t0) / 60, 1)
pipeline.print_summary(metrics)
print(f"\n  Thời gian: {metrics['minutes']} phút")

## §5. Lỗi ở nấc này trông như thế nào

Xem vài ca sai để biết prompt tối giản hụt ở đâu — đây là căn cứ để thiết kế prompt ở nấc 2.

In [ ]:
_bad = [r for r in rows if not r["ea"]]
print(f"  {len(_bad)}/{len(rows)} mẫu sai. Phân bố kiểu lỗi:")
for k, v in sorted(Counter(r["outcome"] for r in _bad).items(), key=lambda x: -x[1]):
    print(f"    {k:<30}{v:>5}")

print(f"\n{'═'*80}\n  5 CA SAI ĐẦU TIÊN\n{'═'*80}")
for r in _bad[:5]:
    print(f"\n  [{r['id']}] {r['question'][:86]}")
    print(f"    gold      : {r['gold_program']}  = {r['gold_answer']}")
    print(f"    dự đoán   : {r['final_program'] or '(không sinh được program)'}  = {r['pred_value']}")

## §6. Lưu kết quả

In [ ]:
save_stage("01_plain", rows, metrics, extra={"prompt_level": "plain", "self_eval": False})
print(f"\n  EA = {metrics['EA']:.4f} | PA_strict = {metrics['PA_strict']:.4f} "
      f"| PA_loose = {metrics['PA_loose']:.4f}")
print("  → Đây là MỐC DƯỚI. Nấc 2 sẽ so với chính con số này.")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  KHỐI ĐỂ GỬI ĐI ĐỐI CHIẾU — bôi đen từ dòng ─── tới hết rồi copy         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
print("\n" + "─" * 74)
print(json.dumps(json.load(open(stage_path("01_plain", "meta"), encoding="utf-8")),
                 ensure_ascii=False, indent=1))
print("─" * 74)


## Kết luận nấc 1

Con số ở đây là mốc dưới của lộ trình. Hai chỉ báo đáng chú ý ngoài EA/PA:

* **`no_program`** — tỉ lệ model không sinh nổi khối `plaintext`. Cao nghĩa là prompt tối
  giản chưa ràng buộc đủ chặt về định dạng.
* **`program_khong_chay_duoc`** — sinh được nhưng executor từ chối (phép lồng nhau, tham
  chiếu `#N` sai, chia 0). Đây chính là nhóm lỗi mà prompt engineering ở nấc 2 nhắm tới.

**Tiếp theo:** `02_prompt_engineering.ipynb`.